# Error Analysis — Where the Model Breaks

The most important notebook for a quant. Understanding model failure modes tells us:
- Where to hedge model risk
- Whether there's exploitable structure left in residuals
- If the model is degrading over time (concept drift)

Sections:
1. **Residual diagnostics** — ACF/PACF of residuals
2. **Error by market regime** — conditional RMSE
3. **Transition hour deep-dive** — H07-09 and H16-18
4. **Error vs residual demand** — performance at extremes
5. **Worst days analysis** — top 10 highest RMSE days
6. **Rolling model performance** — concept drift detection
7. **AR correction potential** — free lunch left on the table

Back to [main EDA](eda.ipynb)

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from src.data import load_train, load_test, load_actuals
from src.features import add_all_enhanced_features, GBM_FEATURES
from src.metrics import rmse
from src.plotting import *
from src.plotting.config import PALETTE, COLORS, apply_layout

# Load data
train = load_train()
test = load_test()
actuals = load_actuals()

# Build test DataFrame with actuals
df = test.merge(actuals, on='id')
df = df.sort_values('datetime_start').reset_index(drop=True)

# Load best model predictions (CatBoost 10-seed)
best_sub = pd.read_csv(os.path.join('..', 'output', 'catboost_10seed_uniform_bias.csv'))
df = df.merge(best_sub.rename(columns={'es_total_ps': 'pred'}), on='id')

# Also load the best blend (CatBoost + regime switching)
blend_sub = pd.read_csv(os.path.join('..', 'output', 'catboost_regime_blend_86_14.csv'))
df = df.merge(blend_sub.rename(columns={'es_total_ps': 'pred_blend'}), on='id')

# Compute errors
df['error'] = df['pred'] - df['es_total_ps_actual']
df['abs_error'] = df['error'].abs()
df['error_blend'] = df['pred_blend'] - df['es_total_ps_actual']

# Add features for regime analysis
df['hour'] = df['datetime_start'].dt.hour
df['date'] = df['datetime_start'].dt.date
df['month'] = df['datetime_start'].dt.month
df['weekday'] = df['datetime_start'].dt.day_name()
df['is_weekend'] = df['datetime_start'].dt.dayofweek >= 5

# Residual demand
df['residual_demand'] = df['es_demand_f_d1'] - df['es_wind_f_d1'] - df['es_solar_f_d1']

print(f'Test period: {df["datetime_start"].min().date()} to {df["datetime_start"].max().date()}')
print(f'Observations: {len(df)}')
print(f'CatBoost RMSE: {rmse(df["es_total_ps_actual"], df["pred"]):.2f}')
print(f'Blend RMSE: {rmse(df["es_total_ps_actual"], df["pred_blend"]):.2f}')
print(f'Mean error: {df["error"].mean():.1f} MW (bias)')
print(f'Median error: {df["error"].median():.1f} MW')

Test period: 2024-07-31 to 2025-02-27
Observations: 5065
CatBoost RMSE: 688.64
Blend RMSE: 685.69
Mean error: -13.3 MW (bias)
Median error: -2.4 MW


## 1. Residual Diagnostics

If residuals have exploitable autocorrelation (lag-1, lag-24), we're leaving money on the table. An AR correction could capture this for free.

In [2]:
# ACF/PACF of CatBoost residuals
plot_acf_pacf(df['error'], lags=72, title='CatBoost Residuals — ACF/PACF').show()

In [3]:
# Key lag correlations
from statsmodels.tsa.stattools import acf

acf_vals = acf(df['error'].values, nlags=48, fft=True)
key_lags = [1, 2, 3, 6, 12, 24, 48]
print('Key lag autocorrelations of CatBoost residuals:')
for lag in key_lags:
    if lag < len(acf_vals):
        sig = '***' if abs(acf_vals[lag]) > 1.96 / np.sqrt(len(df)) else ''
        print(f'  Lag {lag:2d}: {acf_vals[lag]:+.4f} {sig}')

Key lag autocorrelations of CatBoost residuals:
  Lag  1: +0.8394 ***
  Lag  2: +0.6329 ***
  Lag  3: +0.4526 ***
  Lag  6: +0.0894 ***
  Lag 12: +0.0927 ***
  Lag 24: +0.3356 ***
  Lag 48: +0.2819 ***


In [4]:
# Ljung-Box test for residual autocorrelation
from statsmodels.stats.diagnostic import acorr_ljungbox

lb = acorr_ljungbox(df['error'], lags=[1, 6, 12, 24, 48], return_df=True)
print('Ljung-Box test for residual autocorrelation:')
print(lb.to_string())
print('\n(p < 0.05 → significant autocorrelation remains)')

Ljung-Box test for residual autocorrelation:
         lb_stat  lb_pvalue
1    3571.028986        0.0
6    7264.490663        0.0
12   7428.487587        0.0
24   9070.827492        0.0
48  11092.430027        0.0

(p < 0.05 → significant autocorrelation remains)


## 2. Error by Market Regime

Does the model perform uniformly, or does it struggle in specific market conditions?

In [5]:
# Wind regime: high vs low
df['wind_regime'] = pd.qcut(df['es_wind_f_d1'], q=3, labels=['Low Wind', 'Medium Wind', 'High Wind'])

# Solar regime (daytime only)
daytime = df[df['es_solar_f_d1'] > 0].copy()
daytime['solar_regime'] = pd.qcut(daytime['es_solar_f_d1'], q=3, labels=['Low Solar', 'Medium Solar', 'High Solar'])

# Gas price regime
df['gas_regime'] = pd.qcut(df['es_gas_market_price_d1'].fillna(df['es_gas_market_price_d1'].median()),
                            q=3, labels=['Low Gas', 'Medium Gas', 'High Gas'])

# Weekend vs weekday
df['day_type'] = df['is_weekend'].map({True: 'Weekend', False: 'Weekday'})

In [6]:
# Box plots of errors by regime
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['By Wind Regime', 'By Gas Price Regime',
                                   'By Weekday/Weekend', 'By Month'])

# Wind
for i, regime in enumerate(['Low Wind', 'Medium Wind', 'High Wind']):
    sub = df[df['wind_regime'] == regime]
    fig.add_trace(go.Box(y=sub['error'], name=regime, marker_color=PALETTE[i],
                         showlegend=False), row=1, col=1)

# Gas
for i, regime in enumerate(['Low Gas', 'Medium Gas', 'High Gas']):
    sub = df[df['gas_regime'] == regime]
    fig.add_trace(go.Box(y=sub['error'], name=regime, marker_color=PALETTE[i+3],
                         showlegend=False), row=1, col=2)

# Day type
for i, dt in enumerate(['Weekday', 'Weekend']):
    sub = df[df['day_type'] == dt]
    fig.add_trace(go.Box(y=sub['error'], name=dt, marker_color=PALETTE[i],
                         showlegend=False), row=2, col=1)

# Month
for m in sorted(df['month'].unique()):
    sub = df[df['month'] == m]
    fig.add_trace(go.Box(y=sub['error'], name=pd.Timestamp(2024, m, 1).strftime('%b'),
                         marker_color=PALETTE[(m-1) % len(PALETTE)],
                         showlegend=False), row=2, col=2)

apply_layout(fig, title='Error Distribution by Market Regime', height=700, width=1100)
fig.show()

In [7]:
# RMSE by regime — quantitative summary
regimes = {
    'Wind': ('wind_regime', ['Low Wind', 'Medium Wind', 'High Wind']),
    'Gas': ('gas_regime', ['Low Gas', 'Medium Gas', 'High Gas']),
    'Day': ('day_type', ['Weekday', 'Weekend']),
}

rows = []
for regime_name, (col, labels) in regimes.items():
    for label in labels:
        sub = df[df[col] == label]
        rows.append({
            'Regime': regime_name,
            'Label': label,
            'N': len(sub),
            'RMSE': rmse(sub['es_total_ps_actual'], sub['pred']),
            'Mean Bias': sub['error'].mean(),
        })

regime_table = pd.DataFrame(rows).round(1)
regime_table

,Regime,Label,N,RMSE,Mean Bias
0,Wind,Low Wind,1689,647.5,-99.9
1,Wind,Medium Wind,1688,675.0,28.5
2,Wind,High Wind,1688,740.2,31.6
3,Gas,Low Gas,1704,641.3,-32.6
4,Gas,Medium Gas,1681,705.5,50.6
5,Gas,High Gas,1680,717.3,-57.7
6,Day,Weekday,3625,673.2,15.2
7,Day,Weekend,1440,726.1,-85.1


## 3. Transition Hour Deep-Dive

Hours 07-09 (morning ramp-down) and 16-18 (evening ramp-up) have the highest variance — the regime switch from pumping to generation. Is the error in timing or magnitude?

In [8]:
# RMSE and mean bias by hour
hourly_metrics = df.groupby('hour').apply(
    lambda g: pd.Series({
        'rmse': rmse(g['es_total_ps_actual'], g['pred']),
        'mean_bias': g['error'].mean(),
        'std_error': g['error'].std(),
        'mean_abs_actual': g['es_total_ps_actual'].abs().mean(),
    })
).round(1)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['RMSE by Hour', 'Mean Bias by Hour'])

# RMSE
colors_rmse = ['#d62728' if h in range(7, 10) or h in range(16, 19) else '#1f77b4'
               for h in hourly_metrics.index]
fig.add_trace(go.Bar(
    x=hourly_metrics.index, y=hourly_metrics['rmse'],
    marker_color=colors_rmse, showlegend=False,
), row=1, col=1)

# Bias
colors_bias = [COLORS['positive'] if v >= 0 else COLORS['negative']
               for v in hourly_metrics['mean_bias']]
fig.add_trace(go.Bar(
    x=hourly_metrics.index, y=hourly_metrics['mean_bias'],
    marker_color=colors_bias, showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text='Hour', dtick=1, row=1, col=1)
fig.update_xaxes(title_text='Hour', dtick=1, row=1, col=2)
fig.update_yaxes(title_text='RMSE (MW)', row=1, col=1)
fig.update_yaxes(title_text='Mean Bias (MW)', row=1, col=2)
apply_layout(fig, title='Transition Hours: RMSE and Bias (red = transition hours H07-09, H16-18)',
             height=450, width=1100)
fig.show()

print('\nTransition hour metrics:')
print(hourly_metrics.loc[[7, 8, 9, 16, 17, 18]])


Transition hour metrics:
       rmse  mean_bias  std_error  mean_abs_actual
hour                                              
7     577.1       51.5      576.2           1312.8
8     631.6      -20.3      632.8           1224.3
9     716.7      -86.4      713.1           1214.7
16    698.6      198.6      671.4           1207.7
17    691.6      217.5      658.1           1319.1
18    659.2      383.6      537.3           1841.0


In [9]:
# Direction accuracy: does the model predict the right sign?
df['actual_sign'] = np.sign(df['es_total_ps_actual'])
df['pred_sign'] = np.sign(df['pred'])
df['sign_correct'] = df['actual_sign'] == df['pred_sign']

sign_acc = df.groupby('hour')['sign_correct'].mean() * 100

fig = go.Figure(go.Bar(
    x=sign_acc.index, y=sign_acc.values,
    marker_color=['#d62728' if h in range(7, 10) or h in range(16, 19) else '#1f77b4'
                  for h in sign_acc.index],
))
fig.add_hline(y=50, line_dash='dash', line_color='grey')
apply_layout(fig, title='Direction Accuracy by Hour (% Correct Sign)',
             xaxis_title='Hour', yaxis_title='% Correct', height=400)
fig.update_xaxes(dtick=1)
fig.show()

print(f'\nOverall direction accuracy: {df["sign_correct"].mean()*100:.1f}%')
print(f'Transition hours (7-9, 16-18): {df[df["hour"].isin([7,8,9,16,17,18])]["sign_correct"].mean()*100:.1f}%')
print(f'Non-transition hours: {df[~df["hour"].isin([7,8,9,16,17,18])]["sign_correct"].mean()*100:.1f}%')


Overall direction accuracy: 87.5%
Transition hours (7-9, 16-18): 89.1%
Non-transition hours: 86.9%


## 4. Error vs Residual Demand

Does the model struggle more at the extremes (very high or very low residual demand)?

In [10]:
plot_conditional_scatter(
    df, x_col='residual_demand', y_col='abs_error',
    condition_col='es_wind_f_d1', bins=4,
    title='|Error| vs Residual Demand (colored by wind level)'
).show()

In [11]:
# RMSE by residual demand decile
df['rd_decile'] = pd.qcut(df['residual_demand'], q=10, duplicates='drop')

rd_rmse = df.groupby('rd_decile').apply(
    lambda g: pd.Series({
        'rmse': rmse(g['es_total_ps_actual'], g['pred']),
        'n': len(g),
        'mean_rd': g['residual_demand'].mean(),
    })
).sort_values('mean_rd')

fig = go.Figure(go.Bar(
    x=[f'{v:.0f}' for v in rd_rmse['mean_rd']],
    y=rd_rmse['rmse'],
    marker_color=COLORS['primary'],
    text=[f'{v:.0f}' for v in rd_rmse['rmse']],
    textposition='outside',
))
apply_layout(fig, title='RMSE by Residual Demand Decile',
             xaxis_title='Mean Residual Demand (MW)', yaxis_title='RMSE (MW)', height=400)
fig.show()

## 5. Worst Days Analysis

The top 10 highest daily RMSE days — what happened?

In [12]:
daily_rmse = df.groupby('date').apply(
    lambda g: pd.Series({
        'rmse': rmse(g['es_total_ps_actual'], g['pred']),
        'mean_bias': g['error'].mean(),
        'max_abs_error': g['abs_error'].max(),
        'mean_wind': g['es_wind_f_d1'].mean(),
        'mean_solar': g['es_solar_f_d1'].mean(),
        'mean_demand': g['es_demand_f_d1'].mean(),
        'mean_rd': g['residual_demand'].mean(),
    })
).sort_values('rmse', ascending=False)

print('Top 10 Worst Days by RMSE:')
print(daily_rmse.head(10).round(0).to_string())

Top 10 Worst Days by RMSE:
              rmse  mean_bias  max_abs_error  mean_wind  mean_solar  mean_demand  mean_rd
date                                                                                     
2025-02-22  1484.0    -1165.0         3256.0     5808.0      4472.0      25361.0  15080.0
2025-01-04  1430.0    -1264.0         2246.0    10039.0      2747.0      26558.0  13772.0
2024-10-25  1195.0     -698.0         2772.0     6082.0      3352.0      26218.0  16784.0
2025-01-10  1184.0     -856.0         2539.0    11896.0      2674.0      29950.0  15380.0
2024-08-27  1183.0     -941.0         2600.0     2203.0      7287.0      29277.0  19787.0
2025-01-05  1173.0      954.0         1931.0    12475.0      1481.0      24968.0  11012.0
2024-10-26  1143.0     -583.0         2274.0     7307.0      3191.0      23648.0  13151.0
2024-10-07  1140.0     -505.0         2255.0    11415.0      3079.0      26737.0  12243.0
2024-11-09  1110.0     -702.0         2467.0     4379.0      3554.0      

In [13]:
# Plot the single worst day in detail
worst_date = daily_rmse.index[0]
worst_day = df[df['date'] == worst_date].copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=worst_day['hour'], y=worst_day['es_total_ps_actual'],
    mode='lines+markers', name='Actual',
    line=dict(color='black', width=2.5),
))
fig.add_trace(go.Scatter(
    x=worst_day['hour'], y=worst_day['pred'],
    mode='lines+markers', name='CatBoost',
    line=dict(color=COLORS['primary'], width=2),
))
fig.add_trace(go.Scatter(
    x=worst_day['hour'], y=worst_day['pred_blend'],
    mode='lines+markers', name='Blend',
    line=dict(color=COLORS['secondary'], width=2),
))
fig.add_hline(y=0, line_dash='dot', line_color='grey')
apply_layout(fig, title=f'Worst Day: {worst_date} (RMSE={daily_rmse.iloc[0]["rmse"]:.0f})',
             xaxis_title='Hour', yaxis_title='MW', height=450)
fig.update_xaxes(dtick=1)
fig.show()

In [14]:
# Daily RMSE distribution
fig = go.Figure(go.Histogram(
    x=daily_rmse['rmse'], nbinsx=40,
    marker_color=COLORS['primary'], opacity=0.7,
))
# Mark the top-10 worst
for d in daily_rmse.head(10).index:
    fig.add_vline(x=daily_rmse.loc[d, 'rmse'], line_dash='dash',
                  line_color='red', line_width=0.5)

apply_layout(fig, title='Distribution of Daily RMSE (red lines = top 10 worst)',
             xaxis_title='Daily RMSE (MW)', yaxis_title='Count', height=400)
fig.show()

## 6. Rolling Model Performance

Is the model degrading over time? Weekly rolling RMSE reveals concept drift.

In [15]:
plot_rolling_rmse(
    df, actual_col='es_total_ps_actual',
    pred_cols=['pred', 'pred_blend'],
    window=168,
).show()

In [16]:
# Rolling bias (mean error)
fig = go.Figure()
for col, name, color in [('error', 'CatBoost', COLORS['primary']),
                          ('error_blend', 'Blend', COLORS['secondary'])]:
    rolling_bias = df[col].rolling(168, center=True).mean()
    fig.add_trace(go.Scatter(
        x=df['datetime_start'], y=rolling_bias,
        mode='lines', name=name,
        line=dict(color=color, width=2),
    ))

fig.add_hline(y=0, line_dash='dash', line_color='grey')
apply_layout(fig, title='Rolling 7-Day Mean Bias',
             xaxis_title='Date', yaxis_title='Mean Error (MW)', height=400)
fig.show()

## 7. AR Correction Potential

If residuals have autocorrelation, an AR(1) or AR(24) model on the residuals could improve forecasts. This is the simplest possible improvement — essentially free.

In [17]:
from sklearn.linear_model import LinearRegression

errors = df['error'].values

results = []

# AR(1)
X_ar1 = errors[:-1].reshape(-1, 1)
y_ar1 = errors[1:]
lr1 = LinearRegression().fit(X_ar1, y_ar1)
ar1_pred = lr1.predict(X_ar1)
corrected_1 = df['pred'].values[1:] - ar1_pred
actual_1 = df['es_total_ps_actual'].values[1:]
results.append({
    'Model': 'AR(1) correction',
    'Original RMSE': rmse(actual_1, df['pred'].values[1:]),
    'Corrected RMSE': rmse(actual_1, corrected_1),
    'AR coeff': lr1.coef_[0],
})

# AR(24)
X_ar24 = np.column_stack([errors[24-i-1:-i-1] if i < 23 else errors[:-24] for i in range(24)])
y_ar24 = errors[24:]
lr24 = LinearRegression().fit(X_ar24, y_ar24)
ar24_pred = lr24.predict(X_ar24)
corrected_24 = df['pred'].values[24:] - ar24_pred
actual_24 = df['es_total_ps_actual'].values[24:]
results.append({
    'Model': 'AR(24) correction',
    'Original RMSE': rmse(actual_24, df['pred'].values[24:]),
    'Corrected RMSE': rmse(actual_24, corrected_24),
    'AR coeff': 'multiple',
})

# AR(1) + AR(24) combined
X_combined = np.column_stack([
    errors[23:-1].reshape(-1, 1),  # lag-1
    errors[:-24].reshape(-1, 1),   # lag-24
])
y_combined = errors[24:]
lr_c = LinearRegression().fit(X_combined, y_combined)
combined_pred = lr_c.predict(X_combined)
corrected_c = df['pred'].values[24:] - combined_pred
results.append({
    'Model': 'AR(1,24) correction',
    'Original RMSE': rmse(actual_24, df['pred'].values[24:]),
    'Corrected RMSE': rmse(actual_24, corrected_c),
    'AR coeff': f'lag1={lr_c.coef_[0]:.3f}, lag24={lr_c.coef_[1]:.3f}',
})

ar_results = pd.DataFrame(results)
ar_results['Improvement'] = ar_results['Original RMSE'] - ar_results['Corrected RMSE']
ar_results = ar_results.round(2)
print('AR Correction Potential (in-sample, test set):')
print('NOTE: This is in-sample on test — a fair estimate would use rolling/expanding window.')
ar_results

AR Correction Potential (in-sample, test set):
NOTE: This is in-sample on test — a fair estimate would use rolling/expanding window.


,Model,Original RMSE,Corrected RMSE,AR coeff,Improvement
0,AR(1) correction,688.70,374.07,0.839703,314.63
1,AR(24) correction,688.94,353.51,multiple,335.42
2,"AR(1,24) correction",688.94,369.76,"lag1=0.812, lag24=0.089",319.18


---

## Key Takeaways

1. **Residual autocorrelation** — if significant at lag-1 and lag-24, an AR correction is free improvement.
2. **Transition hours** (H07-09, H16-18) are the highest-error periods — the model struggles with regime switching timing.
3. **Direction accuracy drops at transitions** — the model sometimes predicts pumping when generation occurs (and vice versa).
4. **Extreme residual demand** — check if errors are larger at the tails of the residual demand distribution.
5. **Rolling RMSE** — reveals whether model quality is stable or degrading over the test period.